# Applications of Data Science - Summer Term 2026
- Group: 11 - International Taxation
- Group Members:
    - Simon Andreas **ERTL**
    - Berkay **KAYA**
    - Joel **PUTHENPARAMBIL**
    - Fedor **SAMOROKOV**
- Author of this notebook: Simon Andreas **ERTL**
## Group Project: Building an Austrian Tax Law Assistant
This project is split into four parts:
- Creation of a shared corpus regarding Austrian Tax Law from which each group can use for their LLM models. This corpus consists of questions regarding Austrian Tax Law focusing on different topics (e. g. International Taxation), detailed answers to these questions, and sources referenced.
- Applying models to the created corpus. This task itself is split into three parts:
    - Prompt LLMs (Inference): Out-of-the-box Large Language Models should be used to answer the questions created during Task 1.
    - Fine-tune models (Training): The models should then be fine-tuned using additional materials.
    - RAG: Using RAG, the performnance of the models should again be improved.
- Evalution of model performance: This task involves evaluating the performance of the different models by using different evaluation metrics.
- Presentation

This notebook was created in Google Colab using the available T4 GPU architecture. To replicate the results, upload this notebook to Google Colab, and select the right architecture in the top right corner.

### Task 3
For this task, the three results .csv-files and the dataset with the submitted prompt-answer entries have to be imported. For this, the names of the datasets have to be defined first.

In [ ]:
# Name of the reference dataset
reference_dataset = "Austrian Tax Law Dataset.xlsx"

# Names of the model outputs
model1_output = "output_model1_ERTL.csv"
model2_output = "output_model2_ERTL.csv"
model3_output = "output_model3_ERTL.csv"

In the following cell, necessary parameters will be defined.

In [ ]:
COMPUTE_BERTSCORE = True # Decides whether BERTSCORE will be computed (for test runs)
BERTSCORE_MODEL = "bert-base-multilingual-cased" # Setting the model BERTSCORE should use
BERTSCORE_BATCH_SIZE = 8 # Batch size for BERTSCORE
BERTSCORE_DEVICE = "cuda:0" # Setting devide BERTSCORE should run on

In the next steps, the necessary libraries will be imported.

In [ ]:
!pip install bert-score rouge-score nltk -q

In [ ]:
import pandas as pd # For importing datasets
from pathlib import Path # For handling paths
import re # For regular expressions
from collections import Counter # For token overlap counts
import numpy as np # For numerical operations
from bert_score import BERTScorer # For BERT scoring
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction # For BLEU score
from rouge_score import rouge_scorer # For rouge score
from google.colab import files # For downloading the files

The cell below handles the filepath of the input and output files.

In [ ]:
PROJECT_CWD = Path.cwd() # Reading the current working directory

# Dataset containing "gold standard" answers
REFERENCE_DIR = PROJECT_CWD / reference_dataset # Setting the path for the reference dataset

# Datasets containing the predictions of the different models
OUTPUT1_DIR = PROJECT_CWD / model1_output # Setting path for the first output dataset
OUTPUT2_DIR = PROJECT_CWD / model2_output # Setting path for the second output dataset
OUTPUT3_DIR = PROJECT_CWD / model3_output # Setting path for the third output dataset

# Paths for the results
SUMMARY_OUTPUT = PROJECT_CWD / "evaluation_summary_ERTL.csv"
RESULTS_OUTPUT = PROJECT_CWD / "evaluation_results_ERTL.csv"
ERROR_OUTPUT = PROJECT_CWD / "evaluation_error_ERTL.csv"

Next, the datastes will be imported as Pandas dataframes.

In [ ]:
# Importing the Excel file with the correct answers
ref_pd = pd.read_excel(REFERENCE_DIR,
                     sheet_name = "Dataset")

# Only selecting necessary columns
ref_df = ref_pd[["id", "prompt", "correct_answer"]]

In [ ]:
def import_output(path, model_id: str):
  df = pd.read_csv(path) # Read path
  df.insert(0, "model_id", f"MODEL-{model_id}") # Add column with model identifier

  return df # Return modified dataframe

In [ ]:
# Create dictionairy with output dataframes
output_dfs = {"Model 1 - Inference": import_output(OUTPUT1_DIR, "1"),
              "Model 2 - Fine-tuning": import_output(OUTPUT2_DIR, "2"),
              "Model 3 - RAG (fine-tuned)": import_output(OUTPUT3_DIR, "3")}

For a quick check we will see if all dataframes have the same shape.

In [ ]:
print(f"Reference Dataframe: {ref_df.shape}")

for model_name, model_df in output_dfs.items():
  print(f"{model_name}: {model_df.shape}")

As we can see, the reference dataframe has two more entries than the output dataframes, which all have the same shape. This difference has to be consired in the following code (exact matching instead of simple row-wise evaluation).

In the following cells, we are going to define helper functions and functions that will calculate the evaluation metrics.

In [ ]:
# This function normalizes and cleans the text
def normalize_text(text):
  if pd.isna(text):
    return "" # Return empty string if no value given

  text = str(text).strip().lower() # Stringify text, remove excess whitespace and lowercase it
  text = text.replace("\u00a0", " ") # Replace non-breaking space with normal space
  text = re.sub(r"\s+", " ", text) # Reduces multiple spaces into a single space

  return text # Return cleaned text

In [ ]:
# This function splits into tokens
def tokenize(text):
  text = normalize_text(text) # Apply normalizing function

  # The following return extracts words, section symbols, numbers, etc.
  return re.findall(
        r"§+|\d+[a-zA-Z]*|[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*",
        text)

In [ ]:
# This function checks if a reference outputs matches a model output
def exact_match(prediction, reference):
  return int((normalize_text(prediction) == normalize_text(reference)))

In [ ]:
# This function calculates the token-level F1 score
def token_f1(prediction, reference):
  # Tokenize both inputs
  pred_tokens = tokenize(prediction)
  ref_tokens = tokenize(reference)

  # If both are empty, they are exact matches
  if not pred_tokens and not ref_tokens:
    return 1.0

  # If one is empty and the other one not, F1 score of 0.0
  if not pred_tokens or not ref_tokens:
    return 0.0

  # "Counter" counts how often each token appears; "&" takes the minimum count for each shared token
  common_tokens = Counter(pred_tokens) & Counter(ref_tokens)

  overlap = sum(common_tokens.values()) # Sums the amount of common tokens

  # If there is no overlap, they don't match
  if overlap == 0:
    return 0.0

  # Calculate precision -> Of all tokens in the prediction, how many are also in the reference?
  precision = overlap / len(pred_tokens)

  # Of all tokens in the reference, how many did the prediction recover?
  recall = overlap / len(ref_tokens)

  # Calculate and return F1 score -> both precision and recall need to be good for a good F1
  return 2 * precision * recall / (precision + recall)

In [ ]:
# This cell computes the BLEU-4 score

bleu_smoother = SmoothingFunction().method1 # Store smoothing function; otherwise collapses to 0.0 too easily

def bleu_4(prediction, reference):
  # Tokenize
  prediction_tokens = tokenize(prediction)
  reference_tokens = tokenize(reference)

  # If either prediction or reference is empty; return 0 score
  if not prediction_tokens or not reference_tokens:
        return 0.0

  # Apply BLEU function
  return float( # Return as float
    sentence_bleu( # Apply BLEU function
        [reference_tokens], # Expects reference to be wrapped in a list
        prediction_tokens, # Hand over prediction tokens
        weights = (0.25, 0.25, 0.25, 0.25), # Weights of 1-, 2-, 3-, and 4-gram precision
        smoothing_function = bleu_smoother # Apply smoothing function (especially for short answers)
    )
)

In [ ]:
# This cell is for the rouge scores

# Set up scorer, no stemmer
rouge_scorer_instance = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"], # Scores to computer
    use_stemmer = False) # Safer for german language

def rouge_scores(prediction, reference):
    # Normalize text
    normalized_prediction = normalize_text(prediction)
    normalized_reference = normalize_text(reference)

    # If both texts are empty return perfect scores
    if not normalized_prediction and not normalized_reference:
        return {
            "rouge1_f1": 1.0,
            "rouge2_f1": 1.0,
            "rougeL_f1": 1.0}

    # If one text is empty, return 0.0 for all
    if not normalized_prediction or not normalized_reference:
        return {
            "rouge1_f1": 0.0,
            "rouge2_f1": 0.0,
            "rougeL_f1": 0.0}

    # Peform scoring
    score_dict = rouge_scorer_instance.score(
        normalized_reference,
        normalized_prediction)

    # Return F1 scores
    return {
        "rouge1_f1": float(score_dict["rouge1"].fmeasure),
        "rouge2_f1": float(score_dict["rouge2"].fmeasure),
        "rougeL_f1": float(score_dict["rougeL"].fmeasure)}

In [ ]:
# This cell defines the BERTSCORER and the associated function

if COMPUTE_BERTSCORE:
      bertscorer = BERTScorer(
        model_type = BERTSCORE_MODEL, # Set model to use
        lang = "de", # Set language
        batch_size = BERTSCORE_BATCH_SIZE, # Set batch size
        device = BERTSCORE_DEVICE, # Set device
        rescale_with_baseline = False) # Use plain outputs
else:
        bertscorer = None


def compute_bertscore(predictions, references):
    # Normalize text and store as lists
    predictions = [normalize_text(text) for text in predictions]
    references = [normalize_text(text) for text in references]

    # Initialize zeroed arrays
    precision = np.zeros(len(predictions), dtype = float)
    recall = np.zeros(len(predictions), dtype = float)
    f1 = np.zeros(len(predictions), dtype = float)

    # Get indices where entries are not zero
    valid_indices = [
        idx for idx, (prediction, reference) in enumerate(zip(predictions, references))
        if prediction and reference]

    # Return zeroed arrays if no BERTSCORE should be computed
    if not COMPUTE_BERTSCORE:
        return precision, recall, f1

    # Return zeroed arrays if valid indices
    if not valid_indices:
        return precision, recall, f1

    # Get valid answers from valid indices
    valid_predictions = [predictions[idx] for idx in valid_indices]
    valid_references = [references[idx] for idx in valid_indices]

    # Perform precision, recall, and F1 scoring
    p_tensor, r_tensor, f_tensor = bertscorer.score(
        valid_predictions,
        valid_references)

    # Convert Tensors to NumPy arrays
    p_values = p_tensor.detach().cpu().numpy()
    r_values = r_tensor.detach().cpu().numpy()
    f_values = f_tensor.detach().cpu().numpy()

    # Return values to original position (indices were filtered)
    for local_idx, global_idx in enumerate(valid_indices):
        precision[global_idx] = p_values[local_idx]
        recall[global_idx] = r_values[local_idx]
        f1[global_idx] = f_values[local_idx]

    return precision, recall, f1 # Return final arrays


In [ ]:
# This function counts the number of tokens in a answer
def token_count(text):
  return len(tokenize(text))

In [ ]:
# This function is supposed to catch common errros
def classify_error(prediction, reference, exact_score, rouge_l_score, bertscore_f1):
    normalized_prediction = normalize_text(prediction)

    # If prediction and reference are exact matches (probably never)
    if exact_score == 1:
        return "Correct / exact match"

    # If normalized text is empty
    if not normalized_prediction:
        return "Missing answer"

    # If common assistant phrases are found
    if re.search(r"ich verstehe\. du möchtest|stell mir deine frage", normalized_prediction):
        return "Prompt leakage / malformed output"

    # If the prediction is overly uncertain
    if re.search(
        r"die rechtslage ist unklar|die rechtsfolge ist unklar|die antwort ist unklar",
        normalized_prediction):
        return "Overly generic uncertainty"

    # If the answer is too long (2 times longer than reference based on tokens)
    if token_count(prediction) > 2 * max(token_count(reference), 1):
        return "Overly verbose answer"

    if rouge_l_score < 0.1 and bertscore_f1 < 0.75:
        return "Low semantic overlap"

    if rouge_l_score < 0.1 and bertscore_f1 >= 0.75:
        return "Possible paraphrase / low lexical overlap"

    return "No common mistakes found"

Now we will define a loop that will apply calculate the scores for all entries.

In [ ]:
all_results = [] # Collect detailed dataframes here
summary_rows = [] # Collect one summary row per model here

for model_name, output_df in output_dfs.items():
    # Merge reference and output dataframes (left) on id
    merged = ref_df.merge(output_df, on = "id", how = "left")
    merged["answer"] = merged["answer"].fillna("") # Fille na with empty strings

    # Calculate exact_match for every pair
    merged["exact_match"] = [
        exact_match(prediction, reference)
        for prediction, reference in zip(merged["answer"], merged["correct_answer"])]

    # Calculate Token F1 for every pair
    merged["token_f1"] = [
        token_f1(prediction, reference)
        for prediction, reference in zip(merged["answer"], merged["correct_answer"])]

    # Calculate bleu4 for every pair
    merged["bleu4"] = [
        bleu_4(prediction, reference)
        for prediction, reference in zip(merged["answer"], merged["correct_answer"])]

    # Predict rouge scores for all pairs
    rouge_result_rows = [
        rouge_scores(prediction, reference)
        for prediction, reference in zip(merged["answer"], merged["correct_answer"])]

    # Convert dictionairy into dataframe
    rouge_result_df = pd.DataFrame(rouge_result_rows)

    # Extract values into merged dataframe
    merged["rouge1_f1"] = rouge_result_df["rouge1_f1"]
    merged["rouge2_f1"] = rouge_result_df["rouge2_f1"]
    merged["rougeL_f1"] = rouge_result_df["rougeL_f1"]

    # Get length of answers
    merged["answer_length_chars"] = merged["answer"].astype(str).str.len()
    merged["answer_length_tokens"] = merged["answer"].astype(str).map(token_count)

    # Compute BERTSCORE for all entries
    bert_precision, bert_recall, bert_f1 = compute_bertscore(
        merged["answer"].tolist(),
        merged["correct_answer"].tolist())

    # Convert BERTSCORE to dataframe
    merged["bertscore_precision"] = bert_precision
    merged["bertscore_recall"] = bert_recall
    merged["bertscore_f1"] = bert_f1

    merged["error_category"] = [
        classify_error(prediction, reference, exact_score, rouge_l_score, bertscore_f1)
        for prediction, reference, exact_score, rouge_l_score, bertscore_f1 in zip(
            merged["answer"],
            merged["correct_answer"],
            merged["exact_match"],
            merged["rougeL_f1"],
            merged["bertscore_f1"])]

    # Add model name to every entry
    merged["model"] = model_name

    # Add dataframe to results list
    all_results.append(merged)

    # Add summary to summary list
    summary_rows.append({
        "Model": model_name,
        "Rows in output": int(output_df.shape[0]),
        "Rows evaluated": int(merged.shape[0]),
        "Exact Match": merged["exact_match"].mean(),
        "Token F1": merged["token_f1"].mean(),
        "BLEU-4": merged["bleu4"].mean(),
        "ROUGE-1 F1": merged["rouge1_f1"].mean(),
        "ROUGE-2 F1": merged["rouge2_f1"].mean(),
        "ROUGE-L F1": merged["rougeL_f1"].mean(),
        "BERTScore Precision": merged["bertscore_precision"].mean(),
        "BERTScore Recall": merged["bertscore_recall"].mean(),
        "BERTScore F1": merged["bertscore_f1"].mean(),
        "Avg. answer length (chars)": merged["answer_length_chars"].mean(),
        "Avg. answer length (tokens)": merged["answer_length_tokens"].mean()})

Now we will merge the results from all dataframes into a single dataframe.

In [ ]:
results_df = pd.concat(all_results, ignore_index = True)

Now we will create a summary dataframe.

In [ ]:
summary_df = (
    pd.DataFrame(summary_rows)
    .sort_values(
        ["BERTScore F1", "ROUGE-L F1", "BLEU-4", "Exact Match"],
        ascending = False)
    .reset_index(drop = True))

And we will create an additional error summary table.

In [ ]:
error_df = (
    results_df[results_df["error_category"] != "Correct / exact match"]
    .groupby(["model", "error_category"])
    .size()
    .reset_index(name = "count")
    .sort_values(["model", "count"], ascending = [True, False]))

Now the new dataframes can be exported...

In [ ]:
summary_df.to_csv(SUMMARY_OUTPUT, index = False, quotechar = '"')
results_df.to_csv(RESULTS_OUTPUT, index = False, quotechar = '"')
error_df.to_csv(ERROR_OUTPUT, index = False, quotechar = '"')

...and downloaded.

In [ ]:
files.download(SUMMARY_OUTPUT.resolve())

In [ ]:
files.download(RESULTS_OUTPUT.resolve())

In [ ]:
files.download(ERROR_OUTPUT.resolve())

And now we will display the results.

In [ ]:
summary_df

In [ ]:
results_df

In [ ]:
error_df